In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

image = '../src/barred_galaxies/weakly_barred/phys_src_00135.fits'

file = fits.open(image)
sci = file["SCI"].data.astype(float)
err = file["ERR"].data.astype(float)

mean, sky_median, sky_std = sigma_clipped_stats(sci, sigma=3.0)
print(f"sky median = {sky_median:.4f}   sky std = {sky_std:.4f}   peak = {np.nanmax(sci):.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '../src/barred_galaxies/weakly_barred/phys_src_00135.fits'

In [ ]:
file.info()

In [ ]:
sci

In [ ]:
from astropy.convolution import convolve
from photutils.segmentation import make_2dgaussian_kernel
from scipy import ndimage
from matplotlib.patches import Ellipse

# Gentle smoothing (preserves bar ends) used for the mask & fit
kernel_light = make_2dgaussian_kernel(1.5, size=5)
conv = convolve(sci, kernel_light)

# Sky stats computed on the smoothed image (self-consistent subtraction)
_, sky_median_conv, sky_std_conv = sigma_clipped_stats(conv, sigma=3.0)
conv_sub = conv - sky_median_conv

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(conv, origin="lower", cmap="grey", vmin=0, vmax=sci.max())
plt.show()

In [ ]:
# ============================================================
#  Triage: is this galaxy a suspected barred spiral?
# ============================================================
from scipy.ndimage import (
    binary_closing, generate_binary_structure, label as nd_label,
    center_of_mass, sum as nd_sum,
)

# --- 1. Noise-aware threshold ------------------------------------------------
# sky_std_conv was computed in cell [5] on the smoothed image.
peak_conv = np.nanmax(conv_sub)
thresh = np.maximum(3.0 * sky_std_conv, 0.53 * peak_conv)
binary = conv_sub > thresh

# --- 2. Close small gaps (dust lanes, bad pixels) ----------------------------
struct = generate_binary_structure(2, 2)          # 8-connectivity
binary = binary_closing(binary, structure=struct, iterations=1)

# --- 3. Keep largest blob plus any nearby companions -------------------------
labels, n = nd_label(binary, structure=struct)

if n == 0:
    print("SKIP: no pixels above threshold")
    suspected_barred = False
    n_mask_pix = 0
    r_eq = 0.0
    x_c = y_c = np.nan
    a_fit = b_fit = q = angle_deg = np.nan
else:
    if n > 1:
        sizes = nd_sum(binary, labels, range(1, n + 1))
        largest = int(np.argmax(sizes)) + 1
        cy, cx = center_of_mass(binary, labels, largest)
        r_keep = 1.5 * np.sqrt(sizes[largest - 1] / np.pi)
        keep = [largest]
        for i in range(1, n + 1):
            if i == largest or sizes[i - 1] < 3:
                continue
            yi, xi = center_of_mass(binary, labels, i)
            if np.hypot(yi - cy, xi - cx) < r_keep:
                keep.append(i)
        binary = np.isin(labels, keep)

    n_mask_pix = int(binary.sum())
    r_eq = float(np.sqrt(n_mask_pix / np.pi))

    # --- 4. Minimum-size gate ------------------------------------------------
    MIN_R_EQ = 4.0        # ~PSF FWHM; tune to your data
    if r_eq < MIN_R_EQ:
        print(f"SKIP: mask too small (r_eq = {r_eq:.2f} px)")
        suspected_barred = False
        x_c = y_c = np.nan
        a_fit = b_fit = q = angle_deg = np.nan
    else:
        # --- 5. S/N-weighted moment-of-inertia ellipse -----------------------
        ys_m, xs_m = np.where(binary)
        w = np.clip(conv_sub[binary], 0.0, None)     # flux weight, drop negatives
        W = w.sum()

        if W <= 0:
            print("SKIP: non-positive total weight")
            suspected_barred = False
            x_c = y_c = np.nan
            a_fit = b_fit = q = angle_deg = np.nan
        else:
            x_c = float((w * xs_m).sum() / W)
            y_c = float((w * ys_m).sum() / W)
            dx = xs_m - x_c
            dy = ys_m - y_c
            mu_xx = float((w * dx * dx).sum() / W)
            mu_yy = float((w * dy * dy).sum() / W)
            mu_xy = float((w * dx * dy).sum() / W)

            cov = np.array([[mu_xx, mu_xy], [mu_xy, mu_yy]])
            eigvals, eigvecs = np.linalg.eigh(cov)   # ascending

            # Shape from eigenvalues (scale-free), size from the mask AREA.
            # This makes a_fit*b_fit == r_eq**2 and removes the flux-weighting bias.
            q_axis = float(np.sqrt(eigvals[0] / eigvals[1]))   # b/a, in (0, 1]
            a_fit = r_eq / np.sqrt(q_axis)
            b_fit = r_eq * np.sqrt(q_axis)
            theta = np.arctan2(eigvecs[1, -1], eigvecs[0, -1])

            if a_fit < b_fit:
                a_fit, b_fit = b_fit, a_fit
                theta += np.pi / 2
            angle_deg = float(np.degrees(theta) % 180.0)
            q = float(b_fit / a_fit)

            # --- 6. Triage decision ------------------------------------------
            cond_elong      = q < 0.60
            cond_not_edgeon = q > 0.25
            cond_size       = n_mask_pix >= 40
            cond_resolved   = a_fit >= 3.0

            suspected_barred = (
                cond_elong and cond_not_edgeon and cond_size and cond_resolved
            )

            # --- 7. Log ------------------------------------------------------
            print(f"  q = b/a      = {q:.3f}")
            print(f"  a = {a_fit:.2f} px, b = {b_fit:.2f} px, PA = {angle_deg:.1f} deg")
            print(f"  n_mask_pix   = {n_mask_pix}, r_eq = {r_eq:.2f} px")
            print(f"  cond: elong={cond_elong}, not_edgeon={cond_not_edgeon}, "
                  f"size={cond_size}, resolved={cond_resolved}")
            print(f"  >>> suspected barred: {suspected_barred}")

# --- 8. Masked image for optional visualization ------------------------------
mask = binary if n_mask_pix > 0 else np.zeros_like(binary, dtype=bool)
clustered = conv_sub.astype(float).copy()
clustered[~mask] = np.nan

In [ ]:
# --- Visualization: smoothed image + mask overlay ----------------------------
if n_mask_pix == 0:
    print("Nothing to plot: empty mask.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(conv, origin="lower", cmap="inferno",
              vmin=0, vmax=sci.max())
    ax.imshow(np.where(mask, 1, np.nan), origin="lower",
              cmap="grey", alpha=0.25)
    ax.set_title(f"mask ({n_mask_pix} px, r_eq={r_eq:.2f} px)")
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Visualization: fitted moment ellipse overlay ----------------------------
if np.isnan(a_fit):
    print("No ellipse to plot (triage skipped or failed).")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(conv, origin="lower", cmap="inferno",
              vmin=0, vmax=sci.max())
    ax.imshow(np.where(mask, 1, np.nan), origin="lower",
              cmap="spring", alpha=0.25)

    ax.add_patch(Ellipse((x_c, y_c), 2 * a_fit, 2 * b_fit,
                         angle=angle_deg,
                         edgecolor="lime", facecolor="none", linewidth=1.8))
    ax.plot(x_c, y_c, "+", color="cyan",
            markersize=10, markeredgewidth=1.5)

    print(f"center          : ({x_c:.2f}, {y_c:.2f}) px")
    print(f"semi-major a    : {a_fit:.2f} px")
    print(f"semi-minor b    : {b_fit:.2f} px")
    print(f"axis ratio b/a  : {q:.3f}")
    print(f"PA (from +x)    : {angle_deg:.2f} deg")

    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()